# Librerías

In [2]:
import gymnasium as gym # Importa la librería Gymnasium, que se usa para crear y manejar entornos de aprendizaje por refuerzo
import numpy as np # Importa NumPy, una librería para operaciones matemáticas y manejo eficiente de arreglos
from random import randint # Importa la función randint del módulo random para generar números enteros aleatorios
# Configurar NumPy para no usar notación científica al imprimir
np.set_printoptions(suppress=True)

# Ambiente

[Mountain Car](https://gymnasium.farama.org/environments/classic_control/mountain_car/)

![Mountain Car](https://gymnasium.farama.org/_images/mountain_car.gif)

**Mountain Car (Vagón de Montaña)**

El **MDP de Vagón de Montaña (Mountain Car)** es un problema clásico de aprendizaje por refuerzo. Consiste en un coche colocado estocásticamente en el fondo de un valle con forma sinusoidal. El motor del coche no tiene suficiente potencia para subir directamente la colina derecha, por lo que el agente debe aprender a balancearse para acumular inercia.


In [7]:
env= gym.make("MountainCar-v0")
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<MountainCarEnv<MountainCar-v0>>>>>

**Objetivo del MDP**

- Alcanzar la cima de la colina derecha.
- Requiere planificación estratégica: a veces el coche debe moverse inicialmente hacia la colina izquierda para ganar energía suficiente.
- La recompensa usual es negativa en cada paso (-1), incentivando alcanzar la meta en el menor número de pasos posible.
- El episodio termina cuando:
  1. El coche alcanza la cima de la colina derecha (`position >= 0.5`), o
  2. Se alcanza un número máximo de iteraciones.

**Estados (State Space)**

El estado del entorno está definido por **dos variables continuas**:

- **Posición (`position`)**: ubicación del coche en el valle  
  - Rango: `[-1.2, 0.6]`
- **Velocidad (`velocity`)**: velocidad del coche  
  - Rango: `[-0.07, 0.07]`

In [10]:
# Imprime la descripción completa del espacio de observaciones del entorno
print(env.observation_space) # Esto nos dice el tipo de espacio (Box, Discrete, etc.), los valores mínimos y máximos y la forma del estado.

# Imprime los valores mínimos posibles para cada dimensión del estado
# Por ejemplo, en MountainCar-v0: [posición mínima, velocidad mínima]
print("Low:", env.observation_space.low)

# Imprime los valores máximos posibles para cada dimensión del estado
# Por ejemplo, en MountainCar-v0: [posición máxima, velocidad máxima]
print("High:", env.observation_space.high)

# Imprime la forma (shape) del espacio de observaciones
# Esto indica cuántas dimensiones tiene cada estado
# Por ejemplo, (2,) significa que el estado tiene 2 valores (posición y velocidad)
print("Shape:", env.observation_space.shape)

# Imprime el tipo de datos que usa el espacio de observaciones
# Por ejemplo, float32 significa que cada dimensión del estado es un número decimal
print("Dtype:", env.observation_space.dtype)

Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Low: [-1.2  -0.07]
High: [0.6  0.07]
Shape: (2,)
Dtype: float32


**Acciones (Action Space) – Valores Discreta**

El espacio de acciones discreto tiene 3 opciones posibles:

- 0 → Acelerar hacia la izquierda
- 1 → No acelerar
- 2 → Acelerar hacia la derecha

In [12]:
# Imprime la descripción completa del espacio de acciones del entorno
# Esto nos dice el tipo de espacio (Discrete, Box, etc.), los valores mínimos y máximos
print(env.action_space)

# Verificamos si el espacio de acciones es discreto
# Espacios discretos representan acciones como enteros: 0, 1, 2, ...
if isinstance(env.action_space, gym.spaces.Discrete):
    # Imprime el número total de acciones posibles
    print("Número de acciones discretas:", env.action_space.n)
    # Imprime la lista de acciones posibles (enteros)
    print("Acciones posibles:", list(range(env.action_space.n)))
else:
    # Para espacios continuos (Box), imprimimos los rangos mínimos y máximos
    # Esto representa el valor mínimo y máximo que se puede aplicar para cada acción
    print("Acciones continuas - mínimo:", env.action_space.low)
    print("Acciones continuas - máximo:", env.action_space.high)
    # También podemos mostrar la forma y tipo de datos
    print("Shape del espacio de acciones:", env.action_space.shape)
    print("Tipo de datos:", env.action_space.dtype)

Discrete(3)
Número de acciones discretas: 3
Acciones posibles: [0, 1, 2]


La función  <code> discretizar </code> normaliza el estado 'valor' usando los límites del espacio de observaciones
$$\frac{\textit{valor} - \textit{mínimo}}{\textit{máximo} - \textit{mínimo}}$$
- Cada dimensión queda entre $0$ y $1$.

In [14]:
def discretizar(valor):
    aux = ((valor - env.observation_space.low) / 
           (env.observation_space.high - env.observation_space.low)) * 20
    # Multiplicamos por 20 para crear 20 intervalos discretos por dimensión

    # Convertimos los valores a enteros (truncando decimales)
    aux = aux.astype(np.int32)

    # Devolvemos una tupla de enteros
    # Esto permite usarla como índice en una tabla Q o diccionario
    return tuple(aux)

La discretización en <code> MountainCar-v0 </code> se usa porque los estados del entorno son continuos (posición y velocidad), lo que hace imposible utilizar una Q-table directamente.

Al discretizar, se convierten esos valores continuos en un número finito de “intervalos” o estados. Esto permite representar el problema con una tabla finita y aplicar Q-learning de manera práctica.

Además, reduce la complejidad del problema y ayuda al agente a aprender más rápido, ya que agrupa estados similares. Sin embargo, existe un equilibrio: usar pocos intervalos simplifica el aprendizaje pero pierde precisión, mientras que usar muchos mejora la precisión pero aumenta el tiempo de entrenamiento.

**Estado Inicial**
- La posición del coche se asigna a un valor aleatorio uniforme en $[-0,6, -0,4]$.
- La velocidad inicial del coche siempre se asigna a $0$.

In [17]:
discretizar(env.reset()[0]) #Estado inicial

(7, 10)

# Modelo

Crear la Q-table para un entorno con $2$ dimensiones discretizadas y $3$ acciones posibles
- Cada celda $q\_table[i, j, k]$ representa la estimación de la recompensa esperada para el estado discreto $(i, j)$ al tomar la acción $k$

In [20]:
q_table = np.random.uniform(
    low=-1,    # Valor mínimo inicial de la Q (puede ser negativo para exploración)
    high=1,    # Valor máximo inicial de la Q
    size=[20, 20, 3]  # Tamaño de la tabla: 20 niveles para cada dimensión del estado y 3 acciones posibles
)

El carro siempre tiene:

- Una posición
- Una velocidad

Como estos valores pueden ser infinitos (decimales), los simplificamos dividiéndolos en $20$ partes cada uno. Así pasamos de algo complicado a algo manejable:

$20$ posiciones
$20$ velocidades
Es decir, en total $400$ estados posibles

Ahora, en cada situación el carro puede hacer 3 cosas:

- ir a la izquierda
- no hacer nada
- ir a la derecha

Por eso usamos una tabla de tamaño (20, 20, 3)
(es decir, para cada situación guardamos 3 opciones).

**¿Y por qué empezar con valores entre -1 y 1?**

Porque al inicio el carro no sabe nada.

Entonces:

Le damos valores aleatorios: para que pruebe diferentes acciones
Permitimos negativos: porque en el juego hay castigos
Usamos números pequeños (-1 a 1) → para que aprenda de forma estable


**Al inicio son números al azar entre -1 y 1, pero después el agente los va ajustando para reflejar qué acciones son buenas o malas.**

In [22]:
q_table 

array([[[ 0.02063762, -0.58446731,  0.70852277],
        [-0.70125591,  0.95201955, -0.57448018],
        [-0.08901364,  0.65639264, -0.24402397],
        ...,
        [ 0.31159811,  0.74534993, -0.62657295],
        [-0.44656341, -0.85976182,  0.37285111],
        [-0.86488718, -0.21063469, -0.33796493]],

       [[ 0.55433044,  0.42270394, -0.76382182],
        [-0.43217376, -0.56750219,  0.35723587],
        [-0.32369229,  0.86675813, -0.49103852],
        ...,
        [ 0.27157593, -0.987093  ,  0.6971026 ],
        [-0.78038926, -0.18920756, -0.32742894],
        [-0.29194132,  0.57776299, -0.86680522]],

       [[-0.70710229, -0.64384589, -0.50478848],
        [ 0.32288891,  0.99204564, -0.82785999],
        [-0.83677645,  0.3804965 , -0.24586679],
        ...,
        [ 0.23798257, -0.5080745 ,  0.09351651],
        [-0.98931861,  0.13185001, -0.08223653],
        [ 0.84622707,  0.51603945, -0.77087683]],

       ...,

       [[-0.63992956,  0.34616778, -0.002445  ],
        [ 0

In [23]:
len(q_table)

20

In [24]:
q_table.shape

(20, 20, 3)

## Ejemplo

Ejemplo de uso con un estado continuo. Supongamos que 

state_continuo = [posición, velocidad]

In [27]:
state_continuo = np.array([-0.3, 0.02])
state_continuo

array([-0.3 ,  0.02])

Ese valor no lo puedes usar directamente en la Q-table porque es continuo, así que lo discretizas.

Convertimos el estado continuo a discreto

In [29]:
state_discreto = discretizar(state_continuo)  
state_discreto

(10, 12)

**¿Qué significa eso?**
- $12$: Intervalo de posición
- $10$: Intervalo de velocidad

Ya tienes un índice válido para la Q-table.

In [31]:
q_value = q_table[state_discreto]

# Obtener la mejor acción (la de mayor valor Q)
mejor_accion = np.argmax(q_value)

print(f"Valor Q del estado {state_discreto}: {q_value} | Mejor acción: {mejor_accion}")

Valor Q del estado (10, 12): [ 0.23489526 -0.46612389  0.00284759] | Mejor acción: 0


In [32]:
accion = 2
nuevo_estado, recompensa, f1, f2, info = env.step(accion)
print(f"Acción: {accion} | Estado: {nuevo_estado} | Recompensa: {recompensa} | Done: {f1 or f2}")

Acción: 2 | Estado: [-0.49340832  0.00077971] | Recompensa: -1.0 | Done: False


## Entrenamiento

In [34]:
# ==============================
# PARÁMETROS DE Q-LEARNING
# ==============================

alfa = 0.1        # Tasa de aprendizaje: qué tanto se actualizan los valores Q
gamma = 0.95      # Factor de descuento: importancia de recompensas futuras
episodios = 5000  # Número total de episodios de entrenamiento
epsilon = 2       # Parámetro para controlar exploración vs explotación

# Lista para guardar la recompensa total de cada episodio
lista_recompenzas = []

# ==============================
# ENTRENAMIENTO DEL AGENTE
# ==============================

for episodio in range(episodios):

    # Reinicia el entorno y obtiene el estado inicial (continuo). Luego se discretiza para poder usar la Q-table
    estado = discretizar(env.reset()[0])

    final = False  # Indica si el episodio terminó
    f1 = False     # Bandera de éxito
    f2 = False     # Bandera de fallo

    recompensa_total = 0  # Acumulador de recompensas del episodio

    # Mientras el episodio no termine
    while not final:

        # ==============================
        # ESTRATEGIA EPSILON-GREEDY
        # ==============================

        # randint(0,10) genera números del 0 al 10 (11 valores posibles)
        # epsilon = 2, entonces:
        # {0,1,2} → exploración (acción aleatoria) → 3/11 ≈ 27%
        # {3,...,10} → explotación (mejor acción) → 8/11 ≈ 73%

        if randint(0,10) > epsilon:
            # Explotación: elige la mejor acción según la Q-table
            accion = np.argmax(q_table[estado])
        else:
            # Exploración: elige una acción aleatoria
            accion = randint(0,2)

        # ==============================
        # INTERACCIÓN CON EL ENTORNO
        # ==============================

        # Ejecuta la acción y obtiene:
        # nuevo_estado: siguiente estado
        # recompensa: premio o castigo
        # f1, f2: indican si el episodio terminó
        nuevo_estado, recompensa, f1, f2, info = env.step(accion)

        # ==============================
        # ACTUALIZACIÓN Q-LEARNING
        # ==============================

        # Fórmula de Q-learning:
        # Q(s,a) = Q(s,a) + alfa * (recompensa + gamma * max(Q(s',a')) - Q(s,a))

        q_table[estado][accion] = q_table[estado][accion] + alfa * (
            recompensa 
            + gamma * np.max(q_table[discretizar(nuevo_estado)]) 
            - q_table[estado][accion]
        )

        # Actualiza el estado actual
        estado = discretizar(nuevo_estado)

        # Acumula la recompensa
        recompensa_total += recompensa

        # Verifica si el episodio terminó
        final = f1 or f2

    # Guarda la recompensa total del episodio
    lista_recompenzas.append(recompensa_total)

    # Cada 100 episodios imprime el progreso
    if (episodio + 1) % 100 == 0:
        print("Episodio: " + str(episodio) + 
              " | Recompensa promedio: " + str(np.mean(lista_recompenzas)))

# ==============================
# FINALIZAR ENTORNO
# ==============================

env.close()

Episodio: 99 | Recompensa promedio: -200.0
Episodio: 199 | Recompensa promedio: -200.0
Episodio: 299 | Recompensa promedio: -200.0
Episodio: 399 | Recompensa promedio: -200.0
Episodio: 499 | Recompensa promedio: -200.0
Episodio: 599 | Recompensa promedio: -200.0
Episodio: 699 | Recompensa promedio: -199.9557142857143
Episodio: 799 | Recompensa promedio: -199.905
Episodio: 899 | Recompensa promedio: -199.91555555555556
Episodio: 999 | Recompensa promedio: -199.813
Episodio: 1099 | Recompensa promedio: -199.59272727272727
Episodio: 1199 | Recompensa promedio: -199.50333333333333
Episodio: 1299 | Recompensa promedio: -199.21923076923076
Episodio: 1399 | Recompensa promedio: -199.115
Episodio: 1499 | Recompensa promedio: -199.07266666666666
Episodio: 1599 | Recompensa promedio: -199.073125
Episodio: 1699 | Recompensa promedio: -199.12058823529412
Episodio: 1799 | Recompensa promedio: -198.80055555555555
Episodio: 1899 | Recompensa promedio: -198.69684210526316
Episodio: 1999 | Recompensa p

In [35]:
# Crear el entorno de MountainCar
# "MountainCar-v0" es el problema donde un carrito debe subir una montaña
# render_mode="human" permite ver la animación en pantalla (el movimiento del carrito)
env = gym.make("MountainCar-v0", render_mode="human")

In [36]:
# Reinicia el entorno y obtiene el estado inicial (continuo)
# Luego se discretiza para poder usar la Q-table
estado = discretizar(env.reset()[0])

# Variables de control del episodio
final = False   # Indica si el episodio terminó
f1 = False      # Bandera de éxito (llegó a la meta)
f2 = False      # Bandera de fallo (se terminó el tiempo)


# Mientras el episodio no termine
while not final:
    
    # Elige la mejor acción según la Q-table (explotación)
    accion = np.argmax(q_table[estado])
      
    # Ejecuta la acción en el entorno
    # Devuelve el nuevo estado, la recompensa y si terminó el episodio
    nuevo_estado, recompensa, f1, f2, info = env.step(accion)

    # Convierte el nuevo estado continuo a discreto
    estado = discretizar(nuevo_estado)
  
    # Verifica si el episodio terminó (éxito o fallo)
    final = f1 or f2

# Cierra el entorno
env.close()